In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append('../')
sys.path.append('../../Share/')
import baseline, config
import self_supervised_v1

import warnings
warnings.filterwarnings('ignore')

def plot_result(result):
    plt.figure(figsize=(10, 5))
    plt.plot(result.history['accuracy'], label='accuracy', marker='o')
    plt.plot(result.history['val_accuracy'], label='val_accuracy', marker='o')
    plt.title('Pre-training Stage')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()


from collections import Counter

def balance_data(X, y):
    # Count samples per class
    class_counts = Counter(y)
    min_count = min(class_counts.values())  # target: balance all to minority count

    indices_list = []

    for label in sorted(class_counts.keys()):
        label_indices = np.where(y == label)[0]
        selected_indices = np.random.choice(label_indices, size=min_count, replace=False)
        indices_list.extend(selected_indices)

    # Shuffle all selected indices
    balanced_indices = np.random.permutation(indices_list)

    # Subset the data
    X_balanced = X[balanced_indices]
    y_balanced = y[balanced_indices]

    return X_balanced, y_balanced



from sklearn.metrics import confusion_matrix
import seaborn as sns
from tensorflow import keras


def heatmap_confusion_matrix(X_test, y_test, model):
    # Predict class labels on the test set
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)  # convert softmax probs to predicted class
    y_true = y_test
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(np.max(y_test)+1), yticklabels=range(np.max(y_test)+1))
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.show()

    return cm



SUBJECT = "Harold"
trainer = baseline.ModelTrainer(config, subject=SUBJECT)

In [ ]:
Threshold_K = 4  # 초기 학습 데이터
X_init, y_init, X_init_test, y_init_test = trainer.return_until_K_data(K=Threshold_K, train_ratio=0.8)
X_init, y_init = balance_data(X_init, y_init)
X_init_test, y_init_test = balance_data(X_init_test, y_init_test)

print(X_init.shape, y_init.shape, X_init_test.shape, y_init_test.shape)

model_K4 = self_supervised_v1.build_model()  # 모델 생성 및 초기 학습
result = model_K4.fit(X_init, y_init, validation_data=(X_init_test, y_init_test), epochs=100, batch_size=128, verbose=0)
plot_result(result)

In [ ]:
model = model_K4
Threshold_K=4

In [ ]:
final_session = len(config.Info_sub_H2)
# Accuracy 저장 리스트
Same_Session_Test_Acc, Next_Session_Test_Acc = [], []

def evaluate_model(model, data, labels):
    return model.evaluate(data, labels, verbose=0)[1]

# 온라인 학습 루프
for session in range(Threshold_K, final_session-1):
    # 현재 세션 데이터
    X_train, _, _, _ = trainer.return_K_th_data_only(K=session, train_ratio=1)
    #balance_data(X_test, y_test)
    # Pseudo-label 생성 및 온라인 업데이트
    pseudo_labels = self_supervised_v1.generate_pseudo_label(model, X_train)
    next_X, next_y, _, _ = trainer.return_K_th_data_only(K=session + 1, train_ratio=1)
    #X_train, pseudo_labels = balance_data(X_train, pseudo_labels)
    #model = self_supervised_v1.online_update(model, X_train, pseudo_labels)
    history = model.fit(X_train, pseudo_labels, validation_data=(next_X, next_y), epochs=20, batch_size=128, verbose=0)
    Next_Session_Test_Acc.append(np.max(history.history['val_accuracy']))

In [ ]:
acc = model.evaluate(next_X, next_y, verbose=0)[1]
print(acc)

cm = heatmap_confusion_matrix(next_X, next_y, model)

In [ ]:
plt.plot(Next_Session_Test_Acc, label='Next Session Test')
plt.plot(Same_Session_Test_Acc, label='Same Session Test')
plt.legend()
plt.show()